# Inspect modeling-ready data

Loads `candidate_detail.parquet`, `features/fingerprints.parquet`, and `features/molformer_embeddings.parquet` from a pipeline + featurization run, and joins them on `candidate_id` for manual inspection.

Edit `OUT_DIR` if your `--output` lives somewhere other than `docs/`.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT_DIR = Path('docs')
CAND_PATH  = OUT_DIR / 'candidate_detail.parquet'
FP_PATH    = OUT_DIR / 'features' / 'fingerprints.parquet'
EMB_PATH   = OUT_DIR / 'features' / 'molformer_embeddings.parquet'

for p in (CAND_PATH, FP_PATH, EMB_PATH):
    print(f'{p}: {"OK" if p.exists() else "MISSING"}')

docs/candidate_detail.parquet: OK
docs/features/fingerprints.parquet: OK
docs/features/molformer_embeddings.parquet: OK


In [3]:
candidates = pd.read_parquet(CAND_PATH)
print('shape:', candidates.shape)
print('canonical SMILES coverage:', candidates['smiles_canonical'].notna().sum(), '/', len(candidates))
print('standardization status counts:')
print(candidates['smiles_standardization_status'].value_counts(dropna=False))
candidates[['candidate_id', 'drug_name', 'indication', 'highest_phase', 'smiles', 'smiles_canonical', 'smiles_standardization_status']].head()

shape: (7460, 50)
canonical SMILES coverage: 4236 / 7460
standardization status counts:
smiles_standardization_status
ok              4236
empty           3223
failed_parse       1
Name: count, dtype: int64


,candidate_id,drug_name,indication,highest_phase,smiles,smiles_canonical,smiles_standardization_status
0,name:test drug__chemically-induced disorders,test drug,heroin dependence,Phase 2,NaN,NaN,empty
1,db:DB00245__chemically-induced disorders,benztropine,cocaine-related disorders,Phase 2,[H][C@]12CC[C@]([H])(C[C@@]([H])(C1)OC(C1=CC=C...,CN1[C@@H]2CC[C@H]1C[C@@H](OC(c1ccccc1)c1ccccc1)C2,ok
2,db:DB00321__fibromyalgia,amitriptyline,fibromyalgia,Phase 4,CN(C)CCC=C1C2=CC=CC=C2CCC2=CC=CC=C12,CN(C)CCC=C1c2ccccc2CCc2ccccc21,ok
3,name:pdgf b ad5__cardiovascular diseases,pdgf-b/ad5,varicose ulcer,Phase 1,NaN,NaN,empty
4,db:DB00904__alcoholism,ondansetron (zofran),alcoholism,Phase 3,CN1C2=C(C3=CC=CC=C13)C(=O)C(CN1C=CN=C1C)CC2,Cc1nccn1CC1CCc2c(c3ccccc3n2C)C1=O,ok


In [4]:
fingerprints = pd.read_parquet(FP_PATH) if FP_PATH.exists() else pd.DataFrame()
print('shape:', fingerprints.shape)
if len(fingerprints):
    sample = fingerprints.iloc[0]
    print('ecfp4 length:', len(sample['ecfp4']), '(expected 2048) — popcount:', int(np.array(sample['ecfp4']).sum()))
    print('maccs length:', len(sample['maccs']), '(expected 167)  — popcount:', int(np.array(sample['maccs']).sum()))
fingerprints.head()

shape: (4236, 3)
ecfp4 length: 2048 (expected 2048) — popcount: 29
maccs length: 167 (expected 167)  — popcount: 36


,candidate_id,ecfp4,maccs
0,db:DB00245__chemically-induced disorders,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,db:DB00321__fibromyalgia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,db:DB00904__alcoholism,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,db:DB00313__blood-borne infections,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,db:DB00238__acquired immunodeficiency syndrome,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [5]:
embeddings = pd.read_parquet(EMB_PATH) if EMB_PATH.exists() else pd.DataFrame()
print('shape:', embeddings.shape)
if len(embeddings):
    sample = np.array(embeddings.iloc[0]['embedding'])
    print('embedding dim:', sample.shape[0], '— mean:', float(sample.mean()), 'std:', float(sample.std()))
    print('model:', embeddings['model'].iloc[0])
embeddings.head()

shape: (4236, 3)
embedding dim: 768 — mean: -0.0015879215119412038 std: 0.5982868801230751
model: ibm/MoLFormer-XL-both-10pct


,candidate_id,embedding,model
0,db:DB00245__chemically-induced disorders,"[0.35962820053100586, 0.34510695934295654, 0.5...",ibm/MoLFormer-XL-both-10pct
1,db:DB00321__fibromyalgia,"[0.5001422762870789, 0.6271580457687378, 0.932...",ibm/MoLFormer-XL-both-10pct
2,db:DB00904__alcoholism,"[-0.22885023057460785, 0.515791118144989, 0.72...",ibm/MoLFormer-XL-both-10pct
3,db:DB00313__blood-borne infections,"[0.12036643177270889, 0.3698476552963257, 0.75...",ibm/MoLFormer-XL-both-10pct
4,db:DB00238__acquired immunodeficiency syndrome,"[-0.683192253112793, 0.6778616309165955, 0.133...",ibm/MoLFormer-XL-both-10pct


In [6]:
joined = candidates.merge(fingerprints, on='candidate_id', how='left') \
                   .merge(embeddings,   on='candidate_id', how='left')
print('joined shape:', joined.shape)
print('rows with all three:', ((joined['smiles_canonical'].notna()) & (joined['ecfp4'].notna()) & (joined['embedding'].notna())).sum())
joined[['candidate_id', 'drug_name', 'indication', 'smiles_canonical', 'ecfp4', 'embedding']].head()

joined shape: (7460, 54)
rows with all three: 4236


,candidate_id,drug_name,indication,smiles_canonical,ecfp4,embedding
0,name:test drug__chemically-induced disorders,test drug,heroin dependence,NaN,NaN,NaN
1,db:DB00245__chemically-induced disorders,benztropine,cocaine-related disorders,CN1[C@@H]2CC[C@H]1C[C@@H](OC(c1ccccc1)c1ccccc1)C2,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.35962820053100586, 0.34510695934295654, 0.5..."
2,db:DB00321__fibromyalgia,amitriptyline,fibromyalgia,CN(C)CCC=C1c2ccccc2CCc2ccccc21,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.5001422762870789, 0.6271580457687378, 0.932..."
3,name:pdgf b ad5__cardiovascular diseases,pdgf-b/ad5,varicose ulcer,NaN,NaN,NaN
4,db:DB00904__alcoholism,ondansetron (zofran),alcoholism,Cc1nccn1CC1CCc2c(c3ccccc3n2C)C1=O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.22885023057460785, 0.515791118144989, 0.72..."


In [7]:
joined.iloc[0]

candidate_id                                name:test drug__chemically-induced disorders
drug_name                                                                      test drug
drug_name_raw                                                                  Test Drug
indication                                                             heroin dependence
highest_phase                                                                    Phase 2
trial_count                                                                            2
trial_ids                                                     [NCT00000331, NCT00000331]
sponsors                                                [University of Colorado, Denver]
earliest_start_date                                                           2002-12-31
latest_completion_date                                                        2002-12-31
drugbank_id                                                                          NaN
mesh_drug            